# Practice 5-1. GraphRAG 환경 설정 — 로컬 LM Studio 연동

책은 구글 코랩 환경에서 `pip install graphrag` 후 OpenAI(`gpt-4-turbo-preview` 등)를 API 키로 바로 연결하고,
그래프 DB를 구글 드라이브에 저장합니다. 이 노트북은 practice3~4와 동일하게 이 워크스페이스(Docker 컨테이너)
안에서, OpenAI 대신 **LM Studio에 로드된 로컬 모델**로 GraphRAG를 구성합니다.

`graphrag`(현재 설치 버전: 3.x)는 `pip install graphrag`로 설치된 시점에 이미지가 재빌드되면 함께 사라지므로,
매 세션 시작 시 재설치가 필요할 수 있습니다. 책의 순서와 동일하게 4단계로 진행합니다.

1. GraphRAG 설치
2. 작업 디렉터리 생성 (`working_directory`, 구글 드라이브 대신 이 저장소 안의 `practice5/working_directory`)
3. `graphrag init`으로 기본 설정 파일 생성
4. `settings.yaml`/`.env`를 LM Studio 로컬 엔드포인트에 맞게 재구성

> **왜 완성 모델(completion model)이 "nothink" 모델인가?** GraphRAG의 커뮤니티 리포트 생성 단계는 LLM에
> JSON 스키마를 강제하는 구조화 출력(structured output)을 요청합니다. 이때 추론(thinking) 모델을 LM Studio에
> 연결하면, 실제 JSON 응답 전체가 `reasoning_content`에 담기고 `content`는 항상 빈 문자열로 반환되는 버그가
> 있어 스키마 검증이 매번 실패합니다(`litellm.JSONSchemaValidationError`). 반면 순수 텍스트 질의응답에서는
> `content`/`reasoning_content`가 정상적으로 분리됩니다 — 구조화 출력과 결합될 때만 발생하는 문제입니다.
> 따라서 이 노트북들은 추론 기능이 꺼진 `unsloth/Qwen3.6-35B-A3B-MTP-GGUF-nothink` 모델을 완성 모델로 사용합니다.

## 1. GraphRAG 설치

In [1]:
!pip install -q graphrag

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
executorch 1.0.1 requires torch<2.10.0,>=2.9.0, but you have torch 2.10.0+cu128 which is incompatible.
descript-audiotools 0.7.2 requires protobuf<3.20,>=3.9.2, but you have protobuf 6.33.6 which is incompatible.
data-designer-unstructured-seed 0.1.0 requires pandas<3,>=2, but you have pandas 3.0.3 which is incompatible.
data-designer-config 0.5.4 requires pandas<3,>=2.3.3, but you have pandas 3.0.3 which is incompatible.
data-designer-config 0.5.4 requires rich<15,>=13.7.1, but you have rich 15.0.0 which is incompatible.


## 0. 환경 설정

In [2]:
import subprocess
import shutil
from pathlib import Path

import yaml
from openai import OpenAI

# --- LM Studio ---
LMSTUDIO_BASE_URL = "http://...:.../v1"  # 실행 환경의 LM Studio 주소로 교체
LMSTUDIO_API_KEY  = "lm-studio"                              # LM Studio는 키 검증 안 함 (더미)

# LM Studio에 로드된 모델명 (환경마다 달라질 수 있으므로 상단에 모아둔다)
LLM_MODEL   = "gemma-4-e2b-it"  # 완성 모델: non-thinking (JSON 스키마 강제 출력 호환)
EMBED_MODEL = "text-embedding-qwen3-embedding-8b"                       # 임베딩 모델
SEARCH_MAX_CONTEXT_TOKENS = 10_000  # 답변 생성 토큰 여유를 남긴 검색 컨텍스트 한도

# --- 입력 파일 / 작업 디렉터리 (노트북 기준 상대경로) ---
DATA_PATH   = "How_to_invest_money.txt"
WORKING_DIR = Path("working_directory")

print("working directory:", WORKING_DIR)

working directory: working_directory


LM Studio 연결과 두 모델(완성/임베딩)이 정상 응답하는지 먼저 확인합니다.

In [3]:
client = OpenAI(base_url=LMSTUDIO_BASE_URL, api_key=LMSTUDIO_API_KEY)

resp = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": "연결 테스트. 'ok'라고만 답하세요."}],
    max_tokens=20,
)
print("LLM  :", resp.choices[0].message.content)

emb = client.embeddings.create(model=EMBED_MODEL, input="연결 테스트")
print("EMBED:", len(emb.data[0].embedding), "차원")

LLM  : ok
EMBED: 4096 차원


## 2. 작업 디렉터리 설정

책은 `pathlib`으로 `working_directory` 폴더를 생성합니다. 이 노트북도 동일하게 `pathlib.Path.mkdir()`을 사용하되,
구글 드라이브가 아니라 이 저장소 안의 `practice5/working_directory`를 그대로 씁니다(재실행 안전: 이미 있으면 그대로 둠).

In [4]:
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print("작업 디렉터리 준비 완료:", WORKING_DIR.resolve())

작업 디렉터리 준비 완료: /workspace/day_05/working_directory


## 3. GraphRAG 초기화

`graphrag init --root ./working_directory`를 실행하면 `settings.yaml`(전체 파이프라인 설정)과 `.env`(API 키 등
환경 변수)가 생성됩니다. `--model`/`--embedding` 옵션으로 기본 모델명을 미리 지정해 두면, 뒤이어 4단계에서
`api_base`만 추가해 LM Studio를 가리키도록 바꾸면 됩니다.

재실행 안전을 위해 `--force`로 항상 기본 템플릿부터 새로 생성하고, LM Studio용 커스터마이징은 다음 4단계에서
코드로(수작업 편집이 아니라) 매번 동일하게 적용합니다 — 그래야 이 노트북을 몇 번을 다시 실행해도 항상 같은
결과가 됩니다.

In [5]:
result = subprocess.run(
    [
        "graphrag", "init",
        "--root", str(WORKING_DIR),
        "--model", LLM_MODEL,
        "--embedding", EMBED_MODEL,
        "--force",  # 재실행 안전: 기존 설정을 항상 기본 템플릿으로 덮어씀
    ],
    capture_output=True, text=True,
)
print("returncode:", result.returncode)
print(result.stdout or "(stdout 없음)")
if result.returncode != 0:
    print(result.stderr)

print("생성된 파일:", sorted(p.name for p in WORKING_DIR.iterdir()))

returncode: 0
(stdout 없음)
생성된 파일: ['.env', 'input', 'prompts', 'settings.yaml']


## 4. LM Studio 로컬 설정 반영 — `settings.yaml` / `.env`

책은 `.env`에 오픈AI API 키를 직접 입력합니다. 이 노트북은 LM Studio가 키를 검증하지 않으므로 더미 값을 씁니다.

`settings.yaml`의 `completion_models.default_completion_model` / `embedding_models.default_embedding_model`
각각에 `api_base`를 추가해 LM Studio 엔드포인트(`http://host.docker.internal:12345/v1`)를 가리키도록 합니다.
GraphRAG(3.x)는 내부적으로 LiteLLM을 통해 `{model_provider}/{model}` 형식으로 호출하므로, `model_provider: openai`
+ 커스텀 `api_base` 조합만으로 OpenAI 호환 서버(LM Studio)를 그대로 쓸 수 있습니다.

> **`concurrent_requests`를 반드시 1로 낮춰야 하는 이유**: GraphRAG의 기본값은 `concurrent_requests: 25`로,
> 동시에 최대 25개의 요청을 LLM에 보냅니다. 이는 다수의 서버 인스턴스를 두는 OpenAI/Azure에는 적합하지만,
> LM Studio처럼 **단일 모델 인스턴스**가 요청을 처리하는 로컬 환경에서는 요청이 인터리빙되면서 llama.cpp의
> 프롬프트(KV) 캐시가 매 요청마다 다른 컨텍스트로 밀려나(evict) 버립니다. 그 결과 매 요청이 프롬프트를 처음부터
> 다시 처리하게 되어, 청크당 처리 시간이 오히려 수십 배 느려지는 현상이 실제로 발생했습니다(청크당 수십 초 →
> 10분 이상). 로컬 단일 모델 서빙에서는 `concurrent_requests: 1`로 직렬화하는 것이 훨씬 빠릅니다.

Local/Global Search의 컨텍스트는 10,000토큰으로 제한해 모델이 답변을 생성할 토큰 여유를 보장합니다.

In [ ]:
settings_path = WORKING_DIR / "settings.yaml"
env_path = WORKING_DIR / ".env"

# --- .env: LM Studio는 API 키를 검증하지 않으므로 더미 값 사용 ---
env_path.write_text("GRAPHRAG_API_KEY=lm-studio\n", encoding="utf-8")

# --- settings.yaml: completion/embedding 모델에 LM Studio api_base 주입 ---
with open(settings_path, "r", encoding="utf-8") as f:
    settings = yaml.safe_load(f)

settings["completion_models"]["default_completion_model"]["api_base"] = LMSTUDIO_BASE_URL
settings["embedding_models"]["default_embedding_model"]["api_base"] = LMSTUDIO_BASE_URL

# LM Studio는 단일 모델 인스턴스이므로 동시 요청을 직렬화해야 프롬프트 캐시가 유지되어 훨씬 빠르다.
settings["concurrent_requests"] = 1

# 모델의 전체 컨텍스트를 모두 입력에 쓰지 않도록 검색 컨텍스트를 제한한다.
settings["local_search"]["max_context_tokens"] = SEARCH_MAX_CONTEXT_TOKENS
settings["global_search"]["max_context_tokens"] = SEARCH_MAX_CONTEXT_TOKENS
settings["global_search"]["data_max_tokens"] = SEARCH_MAX_CONTEXT_TOKENS

with open(settings_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(settings, f, allow_unicode=True, sort_keys=False)

print("completion_models  :", settings["completion_models"])
print("embedding_models   :", settings["embedding_models"])
print("concurrent_requests:", settings["concurrent_requests"])
print("search context max :", SEARCH_MAX_CONTEXT_TOKENS)

## 5. 입력 문서 배치

책은 구글 드라이브의 파일을 `shutil.copy`로 `working_directory/input`에 복사합니다(또는 깃허브에서 직접
다운로드). 이 노트북은 `practice5/How_to_invest_money.txt`(조지 게어 헨리, 『How to Invest Money』)를 동일한
방식으로 `working_directory/input`에 복사합니다.

In [7]:
input_dir = WORKING_DIR / "input"
input_dir.mkdir(parents=True, exist_ok=True)

destination_path = input_dir / Path(DATA_PATH).name
# copy(=copyfile+copymode) 대신 copyfile만 사용: 컨테이너 재빌드로 파일 소유자/권한이 바뀌면
# copymode의 chmod 호출이 PermissionError를 낼 수 있어, 내용만 복사하면 되는 이 용도엔 더 안전하다.
shutil.copyfile(DATA_PATH, destination_path)

if destination_path.exists():
    print(f"파일이 {destination_path}에 성공적으로 복사되었습니다")
else:
    print("파일 복사 실패")

print("\nworking_directory 최상위 구성:")
for p in sorted(WORKING_DIR.iterdir()):
    print(" -", p.name + ("/" if p.is_dir() else ""))

파일이 working_directory/input/How_to_invest_money.txt에 성공적으로 복사되었습니다

working_directory 최상위 구성:
 - .env
 - input/
 - prompts/
 - settings.yaml


환경 설정이 끝났습니다. `practice5/working_directory`에는 LM Studio를 가리키는 `settings.yaml`/`.env`와
입력 문서(`input/How_to_invest_money.txt`)가 준비되어 있습니다. 다음 노트북(`practice5-2`)에서 실제로
`graphrag index`를 실행해 지식 그래프를 구축합니다.